In [1]:
"""Дедупликация и сравнение баз Fe-Ni-Co-Al-Cr-Ti (август renumbered vs сентябрь)."""
from pathlib import Path
import re
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Border, Side, Font
from openpyxl.utils import get_column_letter

AUG = "2026-08-Database-Fe-Ni-Co-Al-Cr-Ti_renumbered.xlsx"
SEP = "2026-09-Database-Fe-Ni-Co-Al-Cr-Ti.xlsx"
OUT = Path("outputs/tables")

# приведение сентябрьского файла к виду августовского
RENAME = {"Preporation": "Preparation", "Cooling": "Cooling_after_HT1",
          "Cooling.1": "Cooling_after_Recryst", "Cooling.2": "Cooling_after_Aging1"}
DROP = ["Unnamed: 38", "Дополнительные детали"]
CYR = str.maketrans({"С": "C", "В": "B", "а": "a", "е": "e", "о": "o", "р": "p", "с": "c"})

# Ключ дубликата = все столбцы данных. Paper однозначно задаётся DOI, поэтому исключён вместе с ним.
# Чтобы искать только внутристатейные повторы, уберите "Paper" из кортежа.
KEY_IGNORE = ("xl_row", "Paper", "DOI")

# столбцы для поиска конфликтов результата
ELEMENTS = ["Ni", "Al", "Co", "Cr", "Cu", "Fe", "Ti", "V", "Mo", "Hf", "W", "Ta", "B", "C", "Nb", "Mn"]
TREATMENT = ["Preparation", "HT 1, °C", "HT 1, min", "Cooling_after_HT1",
             "Recrystallization, °C ", "Recrystallization, min", "Cooling_after_Recryst",
             "Aging 1, °C", "Aging 1, min", "Cooling_after_Aging1",
             "Aging 2, °C", "Aging 2, min", "Cold deformation, %"]
RESULTS = ["Solvus Temperature (C)", "Phase composition", "L12 fraction, %"]

# цвета и границы
GREEN = PatternFill("solid", start_color="CDEBC5", end_color="CDEBC5")   # оставлена
RED = PatternFill("solid", start_color="F8CBCB", end_color="F8CBCB")     # удалена
YELLOW = PatternFill("solid", start_color="FFF2B3", end_color="FFF2B3")  # значения различаются
BLUE = PatternFill("solid", start_color="D6E6F8", end_color="D6E6F8")    # значение только в одной строке
THICK, THIN = Side(style="thick", color="000000"), Side(style="thin", color="BFBFBF")


# ---------- загрузка и проверки ----------
def load(path):
    df = pd.read_excel(path)
    # кириллическая «С» в заголовке столбца углерода -> латинская C
    df.columns = [c.translate(CYR) if (not c.isascii() and c.strip() == "С") else c
                  for c in df.columns]
    df = df.rename(columns=RENAME).drop(columns=[c for c in DROP if c in df.columns])
    df.insert(0, "xl_row", df.index + 2)          # номер строки Excel, заголовок = 1
    return df


def check_paper_doi(df):
    """Один номер статьи <-> один DOI, нумерация без пропусков."""
    p2d = df.groupby("Paper")["DOI"].nunique()
    d2p = df.groupby("DOI")["Paper"].nunique()
    papers = sorted(df["Paper"].unique())
    assert (p2d == 1).all(), p2d[p2d > 1]
    assert (d2p == 1).all(), d2p[d2p > 1]
    assert papers == list(range(1, len(papers) + 1)), "в нумерации есть пропуски"


# ---------- нормализация и ключи ----------
def norm_cell(v, nd=4):
    """Числа до nd значащих цифр, текст без лишних пробелов и кириллицы."""
    if pd.isna(v):
        return ""
    if isinstance(v, (int, float, np.number)):
        return f"{float(v):.{nd}g}"               # 100.00000000000001 -> 100
    s = str(v).replace("Heusler(L21)", "L21").translate(CYR)
    return re.sub(r"\s+", " ", s).strip()


def norm_prep(v):
    """Preparation: регистр, дефисы и пробелы не различают режимы."""
    if pd.isna(v):
        return ""
    return re.sub(r"\s+", " ", str(v).lower().replace("-", " ")).strip()


def norm_phase(v):
    """Phase composition: порядок фаз не важен ('BCC + B2' = 'B2 + BCC')."""
    if pd.isna(v):
        return ""
    s = str(v).translate(CYR).replace("Heusler(L21)", "L21")
    return " + ".join(sorted(t.strip() for t in s.split("+") if t.strip()))


def norm_value(v, col):
    if col == "Preparation":
        return norm_prep(v)
    if col == "Phase composition":
        return norm_phase(v)
    return norm_cell(v)


def key_from(df, cols):
    return df[cols].apply(lambda r: "|".join(norm_value(v, c) for c, v in r.items()), axis=1)


def make_key(df, ignore=None):
    ignore = KEY_IGNORE if ignore is None else ignore
    cols = [c for c in df.columns if c not in ignore and not c.startswith("_")]
    return key_from(df, cols), cols


# ---------- дубликаты и сравнение ----------
def dedup(df, ignore=None):
    """Возвращает (unique, removed, groups).

    unique  - таблица без повторов; из группы остаётся строка с наибольшим числом
              заполненных ячеек, при равенстве - с меньшим номером;
    removed - удалённые строки с номером оставленной (kept_xl_row), для внутреннего пользования;
    groups  - группы повторов с номерами строк, статей и DOI.
    """
    df = df.copy()
    df["_key"], cols = make_key(df, ignore)
    df["_filled"] = df[cols].notna().sum(axis=1)
    df = df.sort_values(["_key", "_filled", "xl_row"], ascending=[True, False, True])
    g = df.groupby("_key", sort=False)
    df["_n"] = g["xl_row"].transform("size")
    df["_dup_rows"] = g["xl_row"].transform(lambda s: ",".join(map(str, sorted(s))))
    df["_dup_papers"] = g["Paper"].transform(lambda s: ",".join(map(str, sorted(set(s)))))
    df["_dup_dois"] = g["DOI"].transform(lambda s: "; ".join(dict.fromkeys(s.astype(str))))

    keep = ~df.duplicated("_key", keep="first")
    unique = df[keep].sort_values("xl_row")
    removed = df[~keep].sort_values("xl_row").copy()
    removed["kept_xl_row"] = removed["_key"].map(df[keep].set_index("_key")["xl_row"])
    groups = (df[df["_n"] > 1].drop_duplicates("_key")
              [["_n", "_dup_rows", "_dup_papers", "_dup_dois"]].sort_values("_dup_rows"))
    return unique, removed, groups


def compare(aug_u, sep_u):
    """Сравнение очищенных таблиц по ключу -> (только в авг., только в сент., общие)."""
    cols = ["xl_row", "Paper", "DOI", "_key"]
    m = aug_u[cols].merge(sep_u[cols], on="_key", how="outer",
                          suffixes=("_aug", "_sep"), indicator=True)
    return (m[m["_merge"] == "left_only"].dropna(axis=1, how="all"),
            m[m["_merge"] == "right_only"].dropna(axis=1, how="all"),
            m[m["_merge"] == "both"])


# ---------- отчёт 1: дубликаты ----------
def write_duplicate_report(df, path, ignore=None):
    """Excel-отчёт: оставленная строка зелёная, удалённые красные, каждая группа в жирной рамке."""
    d = df.copy()
    d["_key"], cols = make_key(d, ignore)
    d["_filled"] = d[cols].notna().sum(axis=1)
    d = d.sort_values(["_key", "_filled", "xl_row"], ascending=[True, False, True])
    d["_n"] = d.groupby("_key")["xl_row"].transform("size")
    d = d[d["_n"] > 1].copy()
    d["Статус"] = np.where(d.duplicated("_key", keep="first"), "Удалена", "Оставлена")
    d["_first"] = d.groupby("_key")["xl_row"].transform("min")
    d["_ord"] = (d["Статус"] == "Удалена").astype(int)                 # оставленная строка идёт первой
    d = d.sort_values(["_first", "_ord", "xl_row"])
    d["Группа"] = d.groupby("_key", sort=False).ngroup() + 1
    d = d.rename(columns={"xl_row": "Строка Excel (исходная)"})
    front = ["Группа", "Статус", "Строка Excel (исходная)"]
    rest = [c for c in d.columns if c not in front and not c.startswith("_")]
    rep = d[front + rest].reset_index(drop=True)

    summary = (rep.groupby("Группа")
               .apply(lambda g: pd.Series({
                   "Оставлена (строка)": int(g.loc[g["Статус"] == "Оставлена", front[2]].iloc[0]),
                   "Удалены (строки)": ", ".join(map(str, g.loc[g["Статус"] == "Удалена", front[2]])),
                   "Статьи": ", ".join(map(str, g["Paper"].tolist())),
                   "DOI": "; ".join(dict.fromkeys(g["DOI"].astype(str)))}),
                   include_groups=False).reset_index())

    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        summary.to_excel(xw, sheet_name="Сводка", index=False)
        rep.to_excel(xw, sheet_name="Группы", index=False)

    wb = load_workbook(path)
    ws = wb["Группы"]
    ncol = ws.max_column
    for r in range(1, ws.max_row + 1):
        ws.cell(r, 1).font = Font(bold=(r == 1))
    gcol = rep["Группа"].tolist()
    for i, gid in enumerate(gcol):
        r = i + 2
        first = i == 0 or gcol[i - 1] != gid
        last = i == len(gcol) - 1 or gcol[i + 1] != gid
        fill = GREEN if rep.at[i, "Статус"] == "Оставлена" else RED
        for c in range(1, ncol + 1):
            cell = ws.cell(r, c)
            cell.fill = fill
            cell.border = Border(top=THICK if first else THIN, bottom=THICK if last else THIN,
                                 left=THICK if c == 1 else THIN, right=THICK if c == ncol else THIN)
    ws.freeze_panes = "D2"
    ws.auto_filter.ref = ws.dimensions
    for c in range(1, ncol + 1):
        ws.column_dimensions[get_column_letter(c)].width = 14
    ws.column_dimensions["B"].width = 12
    ws.column_dimensions["C"].width = 22
    wb["Сводка"].freeze_panes = "A2"
    for col, w in zip("ABCDE", (8, 20, 24, 24, 60)):
        wb["Сводка"].column_dimensions[col].width = w
    wb.save(path)
    return summary


# ---------- отчёт 2: одинаковые составы и режимы, разный результат ----------
def find_result_conflicts(df, path):
    """df - таблица после удаления полных дубликатов (aug_u)."""
    d = df[[c for c in df.columns if not c.startswith("_")]].copy()
    d["_ct"] = key_from(d, ELEMENTS + TREATMENT)
    info = []
    for ct, g in d.groupby("_ct", sort=False):
        if len(g) < 2:
            continue
        diff, conflict = [], []
        for c in RESULTS:
            vals = g[c].map(lambda v, c=c: norm_value(v, c))
            if vals.nunique() > 1:
                diff.append(c)
                if vals[vals != ""].nunique() > 1:
                    conflict.append(c)
        if diff:
            info.append({"ct": ct, "rows": g["xl_row"].tolist(), "diff": diff, "conflict": conflict})
    if not info:
        print("Конфликтов результата не найдено")
        return pd.DataFrame()

    info.sort(key=lambda x: min(x["rows"]))
    parts, summary = [], []
    for gid, it in enumerate(info, 1):
        g = d[d["_ct"] == it["ct"]].sort_values("xl_row").copy()
        g.insert(0, "Тип", "конфликт значений" if it["conflict"] else "пропуск в одной из строк")
        g.insert(0, "Группа", gid)
        parts.append(g.drop(columns="_ct"))
        summary.append({"Группа": gid, "Строки Excel": ", ".join(map(str, it["rows"])),
                        "Статьи": ", ".join(map(str, g["Paper"])),
                        "Расходятся": "; ".join(it["diff"]),
                        "Тип": parts[-1]["Тип"].iloc[0]})
    rep = pd.concat(parts, ignore_index=True).rename(columns={"xl_row": "Строка Excel"})
    summary = pd.DataFrame(summary)

    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        summary.to_excel(xw, sheet_name="Сводка", index=False)
        rep.to_excel(xw, sheet_name="Группы", index=False)

    wb = load_workbook(path)
    ws = wb["Группы"]
    ncol = ws.max_column
    colidx = {c: i + 1 for i, c in enumerate(rep.columns)}
    gcol = rep["Группа"].tolist()
    for i, gid in enumerate(gcol):
        r = i + 2
        first = i == 0 or gcol[i - 1] != gid
        last = i == len(gcol) - 1 or gcol[i + 1] != gid
        for c in range(1, ncol + 1):
            ws.cell(r, c).border = Border(top=THICK if first else THIN, bottom=THICK if last else THIN,
                                          left=THICK if c == 1 else THIN, right=THICK if c == ncol else THIN)
    for gid, g in rep.groupby("Группа"):
        it = info[gid - 1]
        for c in it["diff"]:
            fill = YELLOW if c in it["conflict"] else BLUE
            for ridx in g.index:
                ws.cell(ridx + 2, colidx[c]).fill = fill
    ws.freeze_panes = "E2"
    ws.auto_filter.ref = ws.dimensions
    for c in range(1, ncol + 1):
        ws.column_dimensions[get_column_letter(c)].width = 14
    ws.column_dimensions["B"].width = 12
    ws.column_dimensions["C"].width = 24
    wb["Сводка"].freeze_panes = "A2"
    for col, w in zip("ABCDE", (8, 24, 20, 48, 26)):
        wb["Сводка"].column_dimensions[col].width = w
    wb.save(path)
    return summary


if __name__ == "__main__":
    OUT.mkdir(parents=True, exist_ok=True)
    aug, sep = load(AUG), load(SEP)
    check_paper_doi(aug)

    aug_u, aug_removed, aug_groups = dedup(aug)
    sep_u, sep_removed, sep_groups = dedup(sep)
    only_aug, only_sep, both = compare(aug_u, sep_u)

    service = [c for c in aug_u if c.startswith("_") and c != "_dup_dois"]
    aug_u.drop(columns=service).to_excel(OUT / "aug_dedup_clean.xlsx", index=False)
    aug_removed.to_excel(OUT / "INTERNAL_aug_removed_duplicates.xlsx", index=False)
    write_duplicate_report(aug, OUT / "INTERNAL_aug_duplicate_report.xlsx")
    find_result_conflicts(aug_u, OUT / "INTERNAL_aug_result_conflicts.xlsx")
    only_aug.to_excel(OUT / "only_in_aug_after_dedup.xlsx", index=False)
    only_sep.to_excel(OUT / "only_in_sep_after_dedup.xlsx", index=False)

    print(f"Aug: {len(aug)} -> {len(aug_u)} (removed {len(aug_removed)})")
    print(f"Sep: {len(sep)} -> {len(sep_u)} (removed {len(sep_removed)})")
    print(f"only Aug: {len(only_aug)}, only Sep: {len(only_sep)}, common: {len(both)}")

Aug: 406 -> 391 (removed 15)
Sep: 374 -> 374 (removed 0)
only Aug: 24, only Sep: 7, common: 367
